# ECG Atrial Fibrillation Detection with Deep Learning
## PhysioNet / Computing in Cardiology Challenge 2017

**Author:** Martin Ofunrein

---

### How to run this notebook

> **Recommended: Google Colab (free, no setup required)**
>
> 1. Go to [colab.research.google.com](https://colab.research.google.com)
> 2. File → Upload notebook → select this file
> 3. Runtime → Change runtime type → **T4 GPU**
> 4. Runtime → Run all
>
> Colab has TensorFlow, GPU access, and all dependencies pre-installed.
> Training will take ~10-20 minutes with GPU.

---

### Background

**What is an ECG?**
An electrocardiogram (ECG or EKG) is a recording of the heart's electrical activity over time.
Electrodes on the skin pick up tiny electrical signals as the heart beats. The result is a
waveform — the familiar heartbeat trace — that doctors use to diagnose heart conditions.

**What is Atrial Fibrillation (AF)?**
AF is the most common serious heart arrhythmia (irregular heartbeat), affecting ~37 million people
worldwide. Instead of a steady, coordinated rhythm, the upper chambers of the heart (atria)
quiver chaotically. AF dramatically increases stroke risk (5×) and heart failure risk.

The Apple Watch's ECG feature does exactly what this notebook does: it records a single-lead ECG
and runs a deep learning classifier to detect AF in real time.

**The Challenge**
Previous algorithms only classified ECGs as Normal or AF. The PhysioNet 2017 challenge
introduced two additional categories, making it a realistic 4-class problem:

| Label | Class | Description |
|-------|-------|-------------|
|  | Normal | Regular sinus rhythm |
|  | AF | Atrial fibrillation |
|  | Other | Other arrhythmia (noisy-but-real) |
|  | Noisy | Signal too noisy to classify |

**Dataset:** 8,528 single-lead ECG recordings, 30 seconds each, sampled at 300 Hz.


---
## Section 1 — Introduction, Background & Setup

### What is an ECG?
An **electrocardiogram (ECG or EKG)** is a recording of the electrical activity of the heart over time. Each heartbeat produces a characteristic wave pattern. By looking at the shape, rhythm, and timing of these waves, doctors can detect heart problems.

### What is Atrial Fibrillation (AF)?
AF is the most common serious heart rhythm disorder. Instead of beating in a regular, coordinated way, the upper chambers of the heart (atria) quiver chaotically. This leads to an irregular, often rapid heart rate and increases the risk of stroke by up to 5×. Early detection is critical — and that's exactly where machine learning on wearable ECG data (think Apple Watch) can save lives.

### What is the PhysioNet 2017 Challenge?
PhysioNet is a repository of medical data for research. In 2017, they ran a challenge asking teams worldwide to build algorithms to classify short (9–60 second) single-lead ECG recordings into the four classes above. The training set has 8,528 recordings with ground-truth labels.

### Why Deep Learning?
Traditional approaches needed hand-crafted features (heart rate variability, R-peak detection, etc.). Deep learning can learn those features automatically from raw signal data, often achieving state-of-the-art performance.

In [ ]:
# ============================================================
# CELL 1: Install required packages
# ============================================================
# This cell installs any packages that might not be present.
# wfdb  — the official Python library for reading PhysioNet/WFDB format files
# If you already have everything installed, these commands will do nothing harmful.
import subprocess, sys

packages_needed = ['wfdb', 'scikit-learn', 'scipy', 'matplotlib', 'pandas', 'numpy']

for pkg in packages_needed:
    try:
        __import__(pkg if pkg != 'scikit-learn' else 'sklearn')
        print(f'{pkg} already installed.')
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'{pkg} installed.')

# Check TensorFlow separately since import name differs from package name
try:
    import tensorflow as tf
    print(f'TensorFlow already installed: version {tf.__version__}')
except ImportError:
    print('Installing TensorFlow (this may take a few minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow', '-q'])
    print('TensorFlow installed.')

In [ ]:
# ============================================================
# CELL 2: Import all libraries we'll use throughout the notebook
# ============================================================

# --- Standard library ---
import os               # File and folder operations
import urllib.request   # Downloading files from the internet
import zipfile          # Unzipping downloaded archives
import warnings
warnings.filterwarnings('ignore')  # Suppress noisy warnings for cleaner output

# --- Numerical computing ---
import numpy as np      # The fundamental package for numerical arrays in Python
import pandas as pd     # DataFrames — like Excel spreadsheets in Python

# --- Signal processing & ECG reading ---
import wfdb             # Reads PhysioNet WFDB-format files (.mat + .hea)
from scipy.signal import resample  # Resampling signals to different lengths

# --- Machine learning utilities ---
from sklearn.model_selection import train_test_split  # Split data into train/val/test
from sklearn.preprocessing import LabelEncoder        # Convert text labels → numbers
from sklearn.metrics import (
    classification_report,    # Precision, Recall, F1 per class
    confusion_matrix           # Confusion matrix (correct vs. wrong predictions)
)
from sklearn.utils.class_weight import compute_class_weight  # Handle class imbalance

# --- Deep learning ---
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
# layers  — building blocks of neural networks (Conv1D, LSTM, Dense, etc.)
# models  — the container that holds a neural network
# callbacks — actions to take during training (e.g., early stopping)

# --- Plotting ---
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
plt.rcParams['figure.dpi'] = 100  # Make plots a bit larger and sharper

# Set random seeds so results are reproducible (same numbers every run)
np.random.seed(42)
tf.random.set_seed(42)

print('All libraries imported successfully.')
print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version:      {np.__version__}')

In [ ]:
# ============================================================
# CELL 3: Download the PhysioNet 2017 Challenge dataset
# ============================================================
# The dataset lives on PhysioNet's servers.
# We try two approaches:
#   1. wfdb.dl_database() — the clean, official way
#   2. Direct ZIP download — fallback if wfdb download fails
#
# The data will be saved to ./data/training2017/
# Each recording has two files:
#   AXXXXX.mat  — the actual ECG signal (MATLAB format)
#   AXXXXX.hea  — a text header with metadata (sample rate, units, etc.)
# Plus REFERENCE.csv — the ground-truth labels for each recording.

DATA_DIR = './data'          # Where to store downloaded data
TRAINING_DIR = os.path.join(DATA_DIR, 'training2017')  # Sub-folder for training data

os.makedirs(DATA_DIR, exist_ok=True)  # Create the folder if it doesn't exist yet

# Only download if we don't have the data already
ref_csv_path = os.path.join(TRAINING_DIR, 'REFERENCE.csv')

if os.path.exists(ref_csv_path):
    print('Dataset already downloaded. Skipping download.')
else:
    print('Attempting to download via wfdb.dl_database()...')
    try:
        # wfdb can talk directly to the PhysioNet servers and download a whole database
        wfdb.dl_database('challenge-2017/1.0.0', dl_dir=DATA_DIR)
        # After download, the training folder may be nested — check and adjust
        possible_nested = os.path.join(DATA_DIR, 'challenge-2017', '1.0.0', 'training2017')
        if not os.path.exists(TRAINING_DIR) and os.path.exists(possible_nested):
            import shutil
            shutil.move(possible_nested, TRAINING_DIR)
        print('Download via wfdb complete.')
    except Exception as e:
        print(f'wfdb download failed ({e}). Falling back to ZIP download...')
        # Direct ZIP download from PhysioNet
        zip_url = 'https://physionet.org/files/challenge-2017/1.0.0/training2017.zip'
        zip_path = os.path.join(DATA_DIR, 'training2017.zip')
        print(f'Downloading from: {zip_url}')
        print('This may take a few minutes (the file is ~450 MB)...')
        urllib.request.urlretrieve(zip_url, zip_path)
        print('Download complete. Extracting...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DATA_DIR)
        os.remove(zip_path)  # Delete the zip to save disk space
        print('Extraction complete.')

# Double-check the data is there
if os.path.exists(ref_csv_path):
    print(f'REFERENCE.csv found at: {ref_csv_path}')
    n_mat = len([f for f in os.listdir(TRAINING_DIR) if f.endswith('.mat')])
    print(f'Number of .mat signal files: {n_mat}')
else:
    print('ERROR: REFERENCE.csv not found. Please check the download manually.')
    print(f'Expected location: {ref_csv_path}')

---
## Section 2 — Data Loading & Exploratory Data Analysis (EDA)

Before we build any model, we need to **understand our data**. EDA answers questions like:
- How many recordings are there in each class?
- What do the raw ECG signals look like?
- How long are the recordings?

This step is critical — surprises in the data (e.g., severe class imbalance) affect how we design our models.

In [ ]:
# ============================================================
# CELL 4: Load the labels from REFERENCE.csv
# ============================================================
# REFERENCE.csv has two columns, no header:
#   Column 0: recording name (e.g., A00001)
#   Column 1: label (N, A, O, or ~)

ref_df = pd.read_csv(
    ref_csv_path,
    header=None,              # The file has no column names
    names=['record', 'label'] # Give the columns meaningful names
)

print('First few rows of the reference file:')
print(ref_df.head(10))
print(f'\nTotal recordings: {len(ref_df)}')
print('\nClass distribution:')
print(ref_df['label'].value_counts())

# Map the tilde (~) label to the word 'noisy' to avoid confusion with Python operators
# We'll keep the original label column but also add a display-friendly version
label_display_map = {'N': 'Normal', 'A': 'AF', 'O': 'Other', '~': 'Noisy'}
ref_df['label_name'] = ref_df['label'].map(label_display_map)

print('\nClass distribution (human-readable):')
print(ref_df['label_name'].value_counts())

In [ ]:
# ============================================================
# CELL 5: Plot the class distribution as a bar chart
# ============================================================
# Visualizing class imbalance is important because:
# - If 90% of samples are 'Normal', a model that always predicts 'Normal'
#   would get 90% accuracy — but would completely miss every AF case!
# - We need to know this BEFORE training so we can compensate.

fig, ax = plt.subplots(figsize=(8, 5))

counts = ref_df['label_name'].value_counts()
colors = ['#2196F3', '#F44336', '#FF9800', '#9E9E9E']  # Blue, Red, Orange, Gray

bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.8)

# Add the exact count as a label on top of each bar
for bar, count in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,  # Center of bar (x position)
        bar.get_height() + 30,              # Just above the bar (y position)
        f'{count:,}\n({count/len(ref_df)*100:.1f}%)',  # Text to display
        ha='center', va='bottom', fontsize=11
    )

ax.set_title('Class Distribution in PhysioNet 2017 Challenge Dataset', fontsize=14)
ax.set_xlabel('ECG Rhythm Class', fontsize=12)
ax.set_ylabel('Number of Recordings', fontsize=12)
ax.set_ylim(0, counts.max() * 1.2)  # Extra space above tallest bar for labels
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print('Note the class imbalance: Normal recordings far outnumber AF and Noisy.')
print('We will address this using class weights during training.')

In [ ]:
# ============================================================
# CELL 6: Load all ECG signals from .mat files
# ============================================================
# Each .mat file contains the raw voltage measurements from the ECG electrodes.
# wfdb.rdrecord() reads both the .hea header and the .mat data.
# record.p_signal  — the signal array, shape (n_samples, n_channels)
# record.fs        — the sampling frequency in Hz (300 Hz for this dataset)
#
# We load ALL signals first, then preprocess them together.
# This may take a minute — there are 8,528 files to read.

print('Loading ECG signals (this will take ~1-2 minutes)...')

raw_signals = []   # We'll store the numpy array for each recording here
valid_records = [] # Track which records loaded successfully
signal_lengths = [] # Track the length (in samples) of each recording

for idx, row in ref_df.iterrows():
    record_name = row['record']
    record_path = os.path.join(TRAINING_DIR, record_name)
    
    try:
        # wfdb.rdrecord reads the signal and header
        record = wfdb.rdrecord(record_path)
        # p_signal has shape (num_samples, num_leads); we only have 1 lead → flatten to 1D
        signal = record.p_signal[:, 0].astype(np.float32)
        raw_signals.append(signal)
        valid_records.append(idx)
        signal_lengths.append(len(signal))
    except Exception as e:
        # If a file is corrupt or missing, skip it with a warning
        print(f'  Warning: Could not load {record_name}: {e}')

# Filter ref_df to only rows that loaded successfully
ref_df_valid = ref_df.loc[valid_records].reset_index(drop=True)

print(f'Successfully loaded {len(raw_signals)} / {len(ref_df)} recordings.')
print(f'Signal length stats:')
print(f'  Min : {np.min(signal_lengths):,} samples  ({np.min(signal_lengths)/300:.1f} s)')
print(f'  Max : {np.max(signal_lengths):,} samples  ({np.max(signal_lengths)/300:.1f} s)')
print(f'  Mean: {np.mean(signal_lengths):,.0f} samples  ({np.mean(signal_lengths)/300:.1f} s)')
print(f'  Median: {np.median(signal_lengths):,.0f} samples  ({np.median(signal_lengths)/300:.1f} s)')

In [ ]:
# ============================================================
# CELL 7: Plot sample ECG traces — one from each class
# ============================================================
# Seeing raw data helps us intuitively understand what the model must learn.
# Normal ECG: regular, repeating P-QRS-T wave pattern
# AF ECG: no clear P waves, irregular R-R intervals
# Other: various abnormal rhythms
# Noisy: high-frequency noise obscuring the signal

DISPLAY_SECONDS = 10  # Show the first 10 seconds of each sample
DISPLAY_SAMPLES = DISPLAY_SECONDS * 300  # 300 Hz × 10 s = 3000 samples
FS = 300  # Sampling frequency

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=False)
class_order = ['N', 'A', 'O', '~']
class_colors = {'N': '#2196F3', 'A': '#F44336', 'O': '#FF9800', '~': '#9E9E9E'}
class_titles = {'N': 'Normal (N)', 'A': 'Atrial Fibrillation (A)', 
                'O': 'Other Rhythm (O)', '~': 'Noisy (~)'}

for ax, cls in zip(axes, class_order):
    # Find the first recording that belongs to this class
    idx = ref_df_valid[ref_df_valid['label'] == cls].index[0]
    sig = raw_signals[idx]
    
    # Limit to DISPLAY_SAMPLES so all plots have the same x-axis range
    n_show = min(DISPLAY_SAMPLES, len(sig))
    time_axis = np.arange(n_show) / FS  # Convert sample index → time in seconds
    
    ax.plot(time_axis, sig[:n_show], color=class_colors[cls], linewidth=0.8)
    ax.set_title(class_titles[cls], fontsize=12, fontweight='bold')
    ax.set_ylabel('Amplitude (mV)', fontsize=9)
    ax.grid(True, alpha=0.3)  # Light grid helps reading the waveform

axes[-1].set_xlabel('Time (seconds)', fontsize=11)
fig.suptitle('Sample ECG Recordings — One Per Class (first 10 seconds)', 
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('sample_ecg_traces.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 8: Plot the distribution of signal lengths
# ============================================================
# Since recordings have different lengths (some 9 seconds, some 60 seconds),
# we need to standardize them before feeding them to a neural network.
# This plot helps us choose a sensible fixed length for padding/truncation.

fig, ax = plt.subplots(figsize=(10, 4))

# Convert samples to seconds for a more intuitive x-axis
lengths_sec = np.array(signal_lengths) / FS

ax.hist(lengths_sec, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(30, color='red', linestyle='--', linewidth=2, label='30-second cutoff (9000 samples)')

ax.set_title('Distribution of ECG Recording Lengths', fontsize=13)
ax.set_xlabel('Recording Length (seconds)', fontsize=11)
ax.set_ylabel('Number of Recordings', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('signal_length_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

pct_under_30 = np.mean(lengths_sec <= 30) * 100
print(f'{pct_under_30:.1f}% of recordings are 30 seconds or shorter.')
print('We will pad short recordings with zeros and truncate long ones to 9000 samples (30s).')

---
## Section 3 — Preprocessing

Raw data almost never goes directly into a neural network. We need to:
1. **Normalize** the signal amplitude so all recordings are on the same scale
2. **Pad or truncate** to a fixed length (9000 samples = 30 s at 300 Hz)
3. **Encode labels** from text strings to integers (N→0, A→1, O→2, ~→3)
4. **Split** into training, validation, and test sets
5. **Compute class weights** to counteract the class imbalance

In [ ]:
# ============================================================
# CELL 9: Normalize and pad/truncate all signals
# ============================================================
# TARGET_LEN = 9000 means each recording will be exactly 9000 samples long.
# That corresponds to 30 seconds of data at 300 Hz.
#
# NORMALIZATION: We subtract the mean and divide by the standard deviation
# (z-score normalization). This puts all signals on a comparable amplitude scale.
# Without this, a recording from one sensor might have values in the range
# [-0.1, 0.1] while another might be [-2, 2] — the model would struggle.
#
# PADDING: If a recording is shorter than 9000 samples, we add zeros at the end.
# Zeros (after normalization) represent a flat, silent signal.
# TRUNCATION: If longer, we just keep the first 9000 samples.

TARGET_LEN = 9000  # 30 seconds × 300 Hz
EPS = 1e-8         # Small constant to prevent division by zero

def normalize_and_pad(signal, target_len=TARGET_LEN):
    """
    Normalize a 1D ECG signal to zero mean and unit variance,
    then pad or truncate to target_len samples.
    
    Parameters
    ----------
    signal     : 1D numpy array of raw ECG voltage values
    target_len : desired output length in samples
    
    Returns
    -------
    1D numpy array of length target_len, dtype float32
    """
    # Step 1: Remove any NaN values that might have crept in during loading
    signal = np.nan_to_num(signal, nan=0.0)
    
    # Step 2: Z-score normalization
    # mean = average amplitude → subtracted so the signal is centered at 0
    # std  = spread of amplitudes → divided so values typically fall in [-3, 3]
    mean = np.mean(signal)
    std  = np.std(signal)
    signal = (signal - mean) / (std + EPS)
    
    # Step 3: Pad or truncate
    if len(signal) >= target_len:
        # Recording is long enough — just keep the first target_len samples
        signal = signal[:target_len]
    else:
        # Recording is too short — pad with zeros at the end
        pad_len = target_len - len(signal)
        signal = np.concatenate([signal, np.zeros(pad_len, dtype=np.float32)])
    
    return signal.astype(np.float32)

# Apply the function to every recording
print(f'Processing {len(raw_signals)} signals...')
X = np.array([normalize_and_pad(sig) for sig in raw_signals], dtype=np.float32)

# X now has shape (n_recordings, TARGET_LEN)
print(f'X shape: {X.shape}  (recordings × samples)')
print(f'X dtype: {X.dtype}')
print(f'X value range: [{X.min():.2f}, {X.max():.2f}] (most values should be near zero)')

In [ ]:
# ============================================================
# CELL 10: Encode class labels and add channel dimension
# ============================================================
# Neural networks work with numbers, not strings.
# LabelEncoder converts: N → 0, A → 1, O → 2, ~ → 3
# (The exact ordering depends on alphabetical order; we'll track it.)
#
# We also need to add a "channel" dimension to X.
# A 1D CNN expects input of shape (batch_size, timesteps, channels).
# We have 1 channel (single-lead ECG), so we add a trailing dimension of 1.

# --- Encode labels ---
le = LabelEncoder()  # This object remembers the mapping
y = le.fit_transform(ref_df_valid['label'].values)
# le.classes_ shows us what index was assigned to each class
print('Label encoding:')
for idx, cls in enumerate(le.classes_):
    display = label_display_map.get(cls, cls)
    print(f'  {cls} ({display}) → {idx}')

NUM_CLASSES = len(le.classes_)
print(f'\nNumber of classes: {NUM_CLASSES}')

# --- Add channel dimension for CNN ---
# Before: X shape = (8528, 9000)
# After:  X shape = (8528, 9000, 1)  ← Keras Conv1D expects this
X_cnn = X[:, :, np.newaxis]  # np.newaxis inserts a new axis at that position
print(f'\nX shape for CNN input: {X_cnn.shape}')

# For LSTM we keep the shape the same (LSTM also takes 3D input)
X_rnn = X_cnn.copy()  # Same shape — both Conv1D and LSTM use (batch, timesteps, features)
print(f'X shape for RNN input: {X_rnn.shape}')

In [ ]:
# ============================================================
# CELL 11: Train / Validation / Test split (70 / 15 / 15)
# ============================================================
# We divide the dataset into three non-overlapping sets:
#
#   TRAIN (70%) — the model SEES this data and learns from it
#   VAL   (15%) — used DURING training to monitor progress and prevent overfitting
#                 (the model does NOT learn from this; it's a sanity check)
#   TEST  (15%) — used ONLY ONCE at the very end to report final performance
#                 (never looked at during training or model selection)
#
# stratify=y ensures each split has the same class proportions as the full dataset.
# Without stratify, you might accidentally put all the Noisy samples in train
# and none in test.

# First split: separate out the test set (15%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_cnn, y,
    test_size=0.15,
    random_state=42,
    stratify=y  # Preserve class proportions in both halves
)

# Second split: from the remaining 85%, carve out the validation set
# We want 15% of the total, which is 15/85 ≈ 17.6% of the remaining data
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.15/0.85,  # 15% of total
    random_state=42,
    stratify=y_trainval
)

print('Dataset split:')
print(f'  Training   : {len(X_train):,} samples ({len(X_train)/len(X_cnn)*100:.1f}%)')
print(f'  Validation : {len(X_val):,} samples ({len(X_val)/len(X_cnn)*100:.1f}%)')
print(f'  Test       : {len(X_test):,} samples ({len(X_test)/len(X_cnn)*100:.1f}%)')

print('\nClass distribution in training set:')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {le.classes_[u]} ({label_display_map.get(le.classes_[u], le.classes_[u])}): {c:,}')

In [ ]:
# ============================================================
# CELL 12: Compute class weights to handle class imbalance
# ============================================================
# The Normal class has ~5× more samples than the AF class.
# If we train without any correction, the model is tempted to just predict
# 'Normal' for everything — it gets rewarded for it far more often!
#
# CLASS WEIGHTS tell Keras: "mistakes on rare classes cost more."
# Mathematically: weight for class i = (n_samples) / (n_classes × n_samples_in_class_i)
#
# This means the loss for a misclassified AF sample gets multiplied by a larger
# weight, so the model pays more attention to getting AF right.

# compute_class_weight from sklearn automatically calculates these balanced weights
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Keras expects a dictionary {class_index: weight}
class_weight_dict = dict(enumerate(class_weights_array))

print('Class weights (higher = model pays more attention to this class):')
for class_idx, weight in class_weight_dict.items():
    cls = le.classes_[class_idx]
    name = label_display_map.get(cls, cls)
    print(f'  Class {class_idx} ({cls} / {name}): weight = {weight:.4f}')

# Also convert integer labels to one-hot vectors for categorical cross-entropy
# One-hot example for 4 classes:
#   Class 0 (N) → [1, 0, 0, 0]
#   Class 1 (A) → [0, 1, 0, 0]
#   Class 2 (O) → [0, 0, 1, 0]
#   Class 3 (~) → [0, 0, 0, 1]
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=NUM_CLASSES)
y_val_cat   = tf.keras.utils.to_categorical(y_val,   num_classes=NUM_CLASSES)
y_test_cat  = tf.keras.utils.to_categorical(y_test,  num_classes=NUM_CLASSES)

print(f'\nOne-hot encoded label shape: {y_train_cat.shape}')

---
## Section 4 — 1D Convolutional Neural Network (CNN) Model

### What is a 1D Convolution?
A **convolution** is a mathematical operation that slides a small "filter" (a.k.a. kernel) across the signal, computing a dot product at each position. Imagine dragging a magnifying glass across the ECG and asking: "Does this little window of the signal match a QRS spike pattern?"

- Each filter learns to detect one type of local pattern (spikes, slow waves, baseline drift)
- Multiple filters in one layer detect multiple patterns simultaneously
- Stacking several convolutional layers allows the network to detect increasingly complex patterns (heartbeat → rhythm → arrhythmia type)

### Architecture
```
Input (9000×1)
  → Conv1D(32 filters, size=7) → BatchNorm → ReLU → MaxPool(4) → Dropout
  → Conv1D(64 filters, size=5) → BatchNorm → ReLU → MaxPool(4) → Dropout
  → Conv1D(128 filters, size=3) → BatchNorm → ReLU → MaxPool(4) → Dropout
  → Conv1D(256 filters, size=3) → BatchNorm → ReLU → MaxPool(4) → Dropout
  → GlobalAveragePooling1D
  → Dense(128) → ReLU → Dropout
  → Dense(4) → Softmax
```

In [ ]:
# ============================================================
# CELL 13: Define and build the 1D CNN model
# ============================================================

def build_cnn_model(input_length, num_classes):
    """
    Build a 1D Convolutional Neural Network for ECG classification.
    
    Architecture: 4 Conv blocks → GlobalAvgPool → Dense head → Softmax
    
    Parameters
    ----------
    input_length : int  — number of time-steps per sample (9000)
    num_classes  : int  — number of output classes (4)
    
    Returns
    -------
    Compiled Keras Model
    """
    
    # tf.keras.Input defines the shape of one input sample.
    # Shape = (input_length, 1) means a 1D sequence of length 9000 with 1 feature.
    inputs = tf.keras.Input(shape=(input_length, 1), name='ecg_input')
    x = inputs
    
    # ---- Convolutional Block 1 ----
    # Conv1D: 32 filters of width 7 → extracts 32 different local features
    # 'same' padding means the output length equals the input length
    x = layers.Conv1D(32, kernel_size=7, padding='same', activation='relu',
                       name='conv1')(x)
    # BatchNormalization stabilizes training by normalizing each layer's output
    # (reduces sensitivity to weight initialization; lets us train faster)
    x = layers.BatchNormalization()(x)
    # MaxPooling1D reduces the sequence length by taking the maximum in each window.
    # pool_size=4 means every 4 consecutive values become 1 → compresses the signal
    x = layers.MaxPooling1D(pool_size=4)(x)
    # Dropout randomly zeros out 20% of neurons during training → prevents overfitting
    x = layers.Dropout(0.2)(x)
    
    # ---- Convolutional Block 2 ----
    # More filters (64) and smaller kernel — learns more complex, finer patterns
    x = layers.Conv1D(64, kernel_size=5, padding='same', activation='relu',
                       name='conv2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)
    x = layers.Dropout(0.2)(x)
    
    # ---- Convolutional Block 3 ----
    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu',
                       name='conv3')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)
    x = layers.Dropout(0.2)(x)
    
    # ---- Convolutional Block 4 ----
    x = layers.Conv1D(256, kernel_size=3, padding='same', activation='relu',
                       name='conv4')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)
    x = layers.Dropout(0.3)(x)
    
    # ---- Global Average Pooling ----
    # Instead of flattening all remaining values, we take the average of each
    # feature map. This compresses (seq_len, 256) → (256,) regardless of seq_len.
    # Benefits: fewer parameters, less overfitting, handles variable-length inputs.
    x = layers.GlobalAveragePooling1D(name='global_avg_pool')(x)
    
    # ---- Dense Classification Head ----
    # A standard fully-connected layer to combine the learned features
    x = layers.Dense(128, activation='relu', name='dense1')(x)
    x = layers.Dropout(0.4)(x)
    
    # ---- Output Layer ----
    # One neuron per class. Softmax converts raw scores to probabilities that sum to 1.
    # e.g., [0.02, 0.85, 0.08, 0.05] → 85% confident this is AF.
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)
    
    # Wrap inputs and outputs into a Model object
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='CNN_ECG')
    
    # Compile: choose optimizer, loss function, and evaluation metric
    # Adam: an adaptive gradient optimizer — usually works well out of the box
    # categorical_crossentropy: standard loss for multi-class classification with one-hot labels
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build the model
cnn_model = build_cnn_model(TARGET_LEN, NUM_CLASSES)

# Print a summary: layers, output shapes, parameter counts
cnn_model.summary()

In [ ]:
# ============================================================
# CELL 14: Train the CNN model
# ============================================================
# CALLBACKS are functions that Keras calls at the end of each epoch.
# We use two:
#
# 1. EarlyStopping: stop training early if the validation loss hasn't improved
#    for 'patience' epochs in a row. This prevents the model from memorizing
#    the training data (overfitting) by wasting epochs on fruitless updates.
#    restore_best_weights=True: automatically revert to the best checkpoint.
#
# 2. ReduceLROnPlateau: if the validation loss plateaus, reduce the learning rate.
#    Think of it as: "when you stop making progress, take smaller steps."
#    This often helps the model escape local minima and continue improving.

BATCH_SIZE = 32   # Number of samples processed before one gradient update
MAX_EPOCHS = 50   # Upper limit — early stopping will likely kick in before this

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',    # Watch validation loss (not training loss!)
    patience=7,            # Stop if no improvement for 7 consecutive epochs
    restore_best_weights=True  # Roll back to the best epoch's weights
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,    # Multiply learning rate by 0.5 when triggered
    patience=3,    # Wait 3 epochs before reducing
    min_lr=1e-6    # Never go below this learning rate
)

# model.fit() runs the training loop:
# for each epoch:
#   for each batch in training data:
#     1. Forward pass: compute predictions
#     2. Compute loss (how wrong are the predictions?)
#     3. Backward pass (backpropagation): compute gradients
#     4. Update weights using the optimizer
#   evaluate on validation set and report metrics

print('Training CNN model...')
print(f'Training samples: {len(X_train):,}  |  Validation samples: {len(X_val):,}')
print(f'Batch size: {BATCH_SIZE}  |  Max epochs: {MAX_EPOCHS}  |  Early stopping patience: 7')
print()

cnn_history = cnn_model.fit(
    X_train, y_train_cat,          # Training data (input, labels)
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    validation_data=(X_val, y_val_cat),  # Validation data evaluated each epoch
    class_weight=class_weight_dict,       # Adjust loss for class imbalance
    callbacks=[early_stop, reduce_lr],    # Callbacks defined above
    verbose=1                             # Print progress each epoch
)

epochs_trained = len(cnn_history.history['loss'])
best_val_acc = max(cnn_history.history['val_accuracy'])
print(f'\nTraining complete! Ran {epochs_trained} epochs.')
print(f'Best validation accuracy: {best_val_acc*100:.2f}%')

In [ ]:
# ============================================================
# CELL 15: Plot CNN training curves
# ============================================================
# Training curves show us how the model improved over time.
# A HEALTHY training run shows:
#   - Both train and val loss decreasing together
#   - Both train and val accuracy increasing together
#   - Val loss eventually levels off (triggering early stopping)
#
# WARNING SIGNS:
#   - Train loss keeps dropping but val loss starts rising → OVERFITTING
#     (model memorized training data, can't generalize)
#   - Both loss values stay high → UNDERFITTING
#     (model isn't learning; try more capacity or more training)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hist = cnn_history.history
ep = range(1, len(hist['loss']) + 1)

# Left plot: Loss
axes[0].plot(ep, hist['loss'],     'b-o', markersize=4, label='Training Loss')
axes[0].plot(ep, hist['val_loss'], 'r-o', markersize=4, label='Validation Loss')
axes[0].set_title('CNN Model — Loss per Epoch', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Categorical Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right plot: Accuracy
axes[1].plot(ep, hist['accuracy'],     'b-o', markersize=4, label='Training Accuracy')
axes[1].plot(ep, hist['val_accuracy'], 'r-o', markersize=4, label='Validation Accuracy')
axes[1].set_title('CNN Model — Accuracy per Epoch', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('1D CNN Training History', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('cnn_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Section 5 — RNN / LSTM Model

### What is an LSTM?
A **Long Short-Term Memory (LSTM)** network is a type of Recurrent Neural Network (RNN). Unlike a CNN, which looks at local windows of the signal, an LSTM processes the signal **step by step in sequence** — left to right — and maintains an internal "memory" (called the cell state and hidden state) that carries information about what it has seen so far.

Think of it like reading a sentence: the meaning of each word depends on all the words before it. The LSTM's memory allows it to capture long-range dependencies — for example, the interval between two R-peaks that are hundreds of samples apart.

### Why does this matter for ECG?
- AF is characterized by **irregular R-R intervals** — the time between heartbeats varies unpredictably
- A pure CNN looks at local patterns; an LSTM can theoretically capture the global rhythm irregularity
- In practice, both architectures perform well; CNNs are usually faster to train

### Architecture
```
Input (9000×1)
  → Conv1D(32, size=7) + MaxPool(4)   ← first reduce the sequence length
  → Conv1D(64, size=5) + MaxPool(4)   ← compress further before LSTM
  → LSTM(128, return_sequences=True)
  → Dropout
  → LSTM(64)
  → Dropout
  → Dense(64) → ReLU
  → Dense(4) → Softmax
```
The CNN layers at the front act as a **feature extractor** and reduce the 9000-step sequence before passing it to the LSTM, which massively speeds up training.

In [ ]:
# ============================================================
# CELL 16: Define and build the CNN-LSTM (RNN) model
# ============================================================

def build_lstm_model(input_length, num_classes):
    """
    Build a hybrid CNN-LSTM model for ECG classification.
    
    The CNN front-end reduces the 9000-step sequence before the LSTM,
    making training much faster while retaining temporal structure.
    
    Parameters
    ----------
    input_length : int  — number of time-steps (9000)
    num_classes  : int  — number of output classes (4)
    
    Returns
    -------
    Compiled Keras Model
    """
    
    inputs = tf.keras.Input(shape=(input_length, 1), name='ecg_input')
    x = inputs
    
    # ---- CNN Feature Extraction Front-End ----
    # Purpose: compress the 9000-step sequence into a shorter, richer representation
    # before the LSTM. A 9000-step LSTM would take forever to train!
    
    # Block 1: 9000 → 2250 samples
    x = layers.Conv1D(32, kernel_size=7, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)  # 9000 / 4 = 2250
    x = layers.Dropout(0.2)(x)
    
    # Block 2: 2250 → 562 samples  
    x = layers.Conv1D(64, kernel_size=5, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)  # 2250 / 4 = 562
    x = layers.Dropout(0.2)(x)
    
    # ---- LSTM Layers ----
    # LSTM(128): 128 memory units, return_sequences=True means we output
    # a vector at EVERY time step (not just the last one).
    # We need this because the second LSTM layer also processes a sequence.
    x = layers.LSTM(128, return_sequences=True, name='lstm1')(x)
    x = layers.Dropout(0.3)(x)
    
    # LSTM(64): return_sequences=False (default) → only output the FINAL hidden state
    # This collapses the sequence into a single fixed-size vector summarizing the
    # entire recording.
    x = layers.LSTM(64, return_sequences=False, name='lstm2')(x)
    x = layers.Dropout(0.3)(x)
    
    # ---- Dense Classification Head ----
    x = layers.Dense(64, activation='relu', name='dense1')(x)
    x = layers.Dropout(0.3)(x)
    
    # ---- Output Layer ----
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='CNN_LSTM_ECG')
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build the model
lstm_model = build_lstm_model(TARGET_LEN, NUM_CLASSES)
lstm_model.summary()

In [ ]:
# ============================================================
# CELL 17: Train the LSTM model
# ============================================================
# Identical training setup to the CNN for a fair comparison.
# LSTMs are generally slower to train than pure CNNs due to their
# sequential nature — they can't be parallelized as easily.

early_stop_lstm = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

reduce_lr_lstm = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

print('Training CNN-LSTM model...')
print('(Note: LSTM training is slower per-epoch than the CNN — this is expected.)')
print()

lstm_history = lstm_model.fit(
    X_train, y_train_cat,
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weight_dict,
    callbacks=[early_stop_lstm, reduce_lr_lstm],
    verbose=1
)

epochs_trained_lstm = len(lstm_history.history['loss'])
best_val_acc_lstm = max(lstm_history.history['val_accuracy'])
print(f'\nTraining complete! Ran {epochs_trained_lstm} epochs.')
print(f'Best validation accuracy: {best_val_acc_lstm*100:.2f}%')

In [ ]:
# ============================================================
# CELL 18: Plot LSTM training curves
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hist_l = lstm_history.history
ep_l = range(1, len(hist_l['loss']) + 1)

axes[0].plot(ep_l, hist_l['loss'],     'b-o', markersize=4, label='Training Loss')
axes[0].plot(ep_l, hist_l['val_loss'], 'r-o', markersize=4, label='Validation Loss')
axes[0].set_title('CNN-LSTM Model — Loss per Epoch', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Categorical Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_l, hist_l['accuracy'],     'b-o', markersize=4, label='Training Accuracy')
axes[1].plot(ep_l, hist_l['val_accuracy'], 'r-o', markersize=4, label='Validation Accuracy')
axes[1].set_title('CNN-LSTM Model — Accuracy per Epoch', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN-LSTM Training History', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('lstm_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Section 6 — Results & Model Comparison

Now we evaluate both models on the **held-out test set** — data neither model has ever seen.

Key metrics:
- **Accuracy**: fraction of all predictions that are correct (can be misleading for imbalanced data)
- **Precision** (per class): of all times we predicted class X, what fraction was actually class X?
- **Recall** (per class): of all actual class X samples, what fraction did we correctly identify?
- **F1 Score**: harmonic mean of precision and recall — the most balanced single metric
- **Confusion Matrix**: a table showing what the model predicted vs. the true class

In [ ]:
# ============================================================
# CELL 19: Evaluate both models on the test set
# ============================================================

# model.predict() runs the trained model on new data (no gradient updates)
# Output shape: (n_test_samples, 4) — a probability distribution over classes
cnn_probs  = cnn_model.predict(X_test,  batch_size=BATCH_SIZE, verbose=0)
lstm_probs = lstm_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

# np.argmax converts probability distributions to hard class predictions
# e.g., [0.05, 0.85, 0.07, 0.03] → 1 (AF)
cnn_preds  = np.argmax(cnn_probs,  axis=1)
lstm_preds = np.argmax(lstm_probs, axis=1)

# Get the integer ground-truth labels
y_true = y_test  # Already integer-encoded (0,1,2,3)

# Class names in encoded order (for display)
class_names = [f'{le.classes_[i]} ({label_display_map.get(le.classes_[i], le.classes_[i])})' 
               for i in range(NUM_CLASSES)]

print('=' * 60)
print('CNN MODEL — Test Set Results')
print('=' * 60)
cnn_loss, cnn_acc = cnn_model.evaluate(X_test, y_test_cat, batch_size=BATCH_SIZE, verbose=0)
print(f'Test Loss: {cnn_loss:.4f}')
print(f'Test Accuracy: {cnn_acc*100:.2f}%')
print()
print(classification_report(y_true, cnn_preds, target_names=class_names, digits=4))

print('=' * 60)
print('CNN-LSTM MODEL — Test Set Results')
print('=' * 60)
lstm_loss, lstm_acc = lstm_model.evaluate(X_test, y_test_cat, batch_size=BATCH_SIZE, verbose=0)
print(f'Test Loss: {lstm_loss:.4f}')
print(f'Test Accuracy: {lstm_acc*100:.2f}%')
print()
print(classification_report(y_true, lstm_preds, target_names=class_names, digits=4))

In [ ]:
# ============================================================
# CELL 20: Plot confusion matrices for both models
# ============================================================
# A confusion matrix is a square table where:
#   - Rows = True (actual) class
#   - Columns = Predicted class
#   - Diagonal entries = CORRECT predictions (we want these as high as possible)
#   - Off-diagonal entries = ERRORS (misclassifications)
#
# Example of a common error: confusing AF with Other — both have irregular rhythms.

def plot_confusion_matrix(y_true, y_pred, class_names, title, ax):
    """
    Plot a normalized confusion matrix on the given matplotlib axes.
    
    Normalized means each row sums to 1 (i.e., shows proportions rather than counts),
    making it easier to compare classes with very different sample counts.
    """
    cm = confusion_matrix(y_true, y_pred)  # Raw count matrix
    
    # Normalize by dividing each row by the number of true samples in that class
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    # Display the normalized matrix as a heatmap
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    # Add tick labels on both axes
    short_names = [cn.split('(')[1].rstrip(')') if '(' in cn else cn for cn in class_names]
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(short_names, fontsize=10)
    ax.set_yticklabels(short_names, fontsize=10)
    
    # Print values inside each cell
    thresh = cm_norm.max() / 2.0  # Use white text on dark backgrounds for readability
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                    ha='center', va='center',
                    color='white' if cm_norm[i,j] > thresh else 'black',
                    fontsize=9)
    
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('True Label', fontsize=10)
    ax.set_xlabel('Predicted Label', fontsize=10)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_confusion_matrix(y_true, cnn_preds,  class_names, 
                      '1D CNN — Confusion Matrix (normalized)', axes[0])
plot_confusion_matrix(y_true, lstm_preds, class_names, 
                      'CNN-LSTM — Confusion Matrix (normalized)', axes[1])

plt.suptitle('Confusion Matrices — Test Set\n(cell value: proportion, count in parentheses)', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 21: Side-by-side comparison table
# ============================================================
# We summarize the key metrics for both models in one table.
# This is the kind of result table you'd put in a paper or report.

from sklearn.metrics import precision_recall_fscore_support

def get_per_class_metrics(y_true, y_pred):
    """Return precision, recall, f1 per class plus macro-averaged f1."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
    macro_f1 = np.mean(f)
    return p, r, f, macro_f1

cnn_p, cnn_r, cnn_f, cnn_macro_f1 = get_per_class_metrics(y_true, cnn_preds)
lstm_p, lstm_r, lstm_f, lstm_macro_f1 = get_per_class_metrics(y_true, lstm_preds)

# Build a comparison DataFrame
short_class_labels = [le.classes_[i] for i in range(NUM_CLASSES)]

comparison_rows = []
for i, cls in enumerate(short_class_labels):
    comparison_rows.append({
        'Class': cls,
        'CNN Precision': f'{cnn_p[i]:.4f}',
        'CNN Recall':    f'{cnn_r[i]:.4f}',
        'CNN F1':        f'{cnn_f[i]:.4f}',
        'LSTM Precision': f'{lstm_p[i]:.4f}',
        'LSTM Recall':    f'{lstm_r[i]:.4f}',
        'LSTM F1':        f'{lstm_f[i]:.4f}',
    })

# Add summary row
comparison_rows.append({
    'Class': 'OVERALL (Accuracy)',
    'CNN Precision': '',
    'CNN Recall':    '',
    'CNN F1':        f'{cnn_acc*100:.2f}%',
    'LSTM Precision': '',
    'LSTM Recall':    '',
    'LSTM F1':        f'{lstm_acc*100:.2f}%',
})
comparison_rows.append({
    'Class': 'Macro F1',
    'CNN Precision': '',
    'CNN Recall':    '',
    'CNN F1':        f'{cnn_macro_f1:.4f}',
    'LSTM Precision': '',
    'LSTM Recall':    '',
    'LSTM F1':        f'{lstm_macro_f1:.4f}',
})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.set_index('Class', inplace=True)

print('Model Comparison Table (Test Set)')
print('=' * 90)
print(comparison_df.to_string())
print('=' * 90)

In [ ]:
# ============================================================
# CELL 22: F1 Score comparison bar chart
# ============================================================
# A visual comparison of per-class F1 scores is easier to read
# than a table full of numbers.

x = np.arange(NUM_CLASSES)  # x positions for class groups
width = 0.35                 # Width of each bar

fig, ax = plt.subplots(figsize=(10, 6))

bars_cnn  = ax.bar(x - width/2, cnn_f,  width, label='1D CNN',   color='steelblue', edgecolor='black')
bars_lstm = ax.bar(x + width/2, lstm_f, width, label='CNN-LSTM', color='coral',     edgecolor='black')

# Add value labels on top of each bar
for bar in bars_cnn:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars_lstm:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# Class labels for x-axis
full_labels = [f'{le.classes_[i]}\n({label_display_map.get(le.classes_[i], le.classes_[i])})'  
               for i in range(NUM_CLASSES)]

ax.set_xticks(x)
ax.set_xticklabels(full_labels, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Class F1 Score: 1D CNN vs CNN-LSTM (Test Set)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('f1_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nCNN  Macro F1: {cnn_macro_f1:.4f}')
print(f'LSTM Macro F1: {lstm_macro_f1:.4f}')

winner = 'CNN' if cnn_macro_f1 >= lstm_macro_f1 else 'CNN-LSTM'
print(f'\nOverall winner by Macro F1: {winner}')

In [ ]:
# ============================================================
# CELL 23: Discussion — which model performed better and why?
# ============================================================
# This cell prints a programmatic discussion based on the actual results.
# The analysis updates itself based on the numbers computed above.

af_idx = list(le.classes_).index('A')  # Find which integer encodes 'A' (AF class)

print('DISCUSSION')
print('=' * 70)
print()
print(f'Overall Accuracy:')
print(f'  1D CNN    : {cnn_acc*100:.2f}%')
print(f'  CNN-LSTM  : {lstm_acc*100:.2f}%')
print()
print(f'AF (Atrial Fibrillation) Detection — the clinically critical class:')
print(f'  1D CNN    F1 = {cnn_f[af_idx]:.4f}  |  Recall = {cnn_r[af_idx]:.4f}')
print(f'  CNN-LSTM  F1 = {lstm_f[af_idx]:.4f}  |  Recall = {lstm_r[af_idx]:.4f}')
print()

if cnn_macro_f1 > lstm_macro_f1:
    print('The 1D CNN outperforms the CNN-LSTM on macro F1.')
    print('This is consistent with the literature: pure CNNs often match or beat')
    print('LSTMs on ECG tasks because the key discriminative features (QRS morphology,')
    print('R-peak regularity) are captured well by local convolutional filters.')
    print('CNNs are also faster to train and easier to deploy on edge devices.')
else:
    print('The CNN-LSTM outperforms the 1D CNN on macro F1.')
    print('This suggests that temporal long-range dependencies (e.g., R-R interval')
    print('irregularity over many heartbeats) are important discriminative features')
    print('that the LSTM captures better than a purely local convolutional approach.')

print()
print('Class Imbalance Effect:')
print('  Both models benefit from class_weight. Without it, recall on AF and Noisy')
print('  would collapse as the model defaults to predicting Normal.')
print()
print('Hardest Class:')
all_f1 = {'CNN': cnn_f, 'LSTM': lstm_f}
worst_cnn  = le.classes_[np.argmin(cnn_f)]
worst_lstm = le.classes_[np.argmin(lstm_f)]
print(f'  CNN hardest class  : {worst_cnn} (F1 = {min(cnn_f):.4f})')
print(f'  LSTM hardest class : {worst_lstm} (F1 = {min(lstm_f):.4f})')
print('  The Noisy (~) class is typically hardest because it is the smallest class')
print('  and has no consistent signal pattern — by definition, it is noisy.')

---
## Section 7 — Conclusions

### Summary of Findings

In this project, we built and compared two deep learning architectures for classifying short ECG recordings into four rhythm categories using the PhysioNet 2017 Challenge dataset.

**Key takeaways:**

1. **Deep learning on raw ECG is viable.** Both models learned directly from the time-series signal without any hand-crafted features (no R-peak detection, no HRV computation). The network discovered relevant patterns on its own.

2. **Class imbalance is a real problem.** The Normal class is over-represented (~5,000 vs. ~750 AF recordings). Using `class_weight` during training was essential to get meaningful recall on the minority classes.

3. **CNNs are competitive with LSTMs.** 1D CNNs are generally faster to train and can achieve comparable or better accuracy on ECG data. This is because ECG classification often depends on local morphological patterns (e.g., the shape of the QRS complex) that CNNs are well-suited to detect.

4. **AF detection specifically** is the highest-stakes subtask. A high recall for AF means we catch more true positives (fewer missed diagnoses), even at the cost of some precision (a few false alarms).

### Real-World Applications: Apple Watch and Beyond

The Apple Watch Series 4 and later include a single-lead ECG sensor — effectively the same type of data as the PhysioNet challenge. Apple received FDA clearance in 2018 to flag AF, and since then their algorithm has been credited with detecting previously unknown AF in thousands of users.

Models trained on PhysioNet data (or data like it) are the direct ancestors of what runs on these devices. The challenges are real:
- **Edge inference**: the model must run on a low-power processor with millisecond latency
- **Data quality**: wrist ECG is noisier than clinical ECG
- **Regulatory**: FDA approval requires extensive clinical validation

Deep learning has made it possible to embed cardiologist-level rhythm detection into a device you wear on your wrist.

In [ ]:
# ============================================================
# CELL 24: Save trained models to disk
# ============================================================
# Saving the trained models means we don't have to retrain from scratch
# if we want to use them again later (e.g., for inference on new ECG data).
#
# The .keras format saves everything: architecture, weights, optimizer state.

cnn_model.save('cnn_ecg_model.keras')
lstm_model.save('lstm_ecg_model.keras')

print('Models saved:')
print('  cnn_ecg_model.keras')
print('  lstm_ecg_model.keras')
print()
print('To reload a model later:')
print("  model = tf.keras.models.load_model('cnn_ecg_model.keras')")
print("  predictions = model.predict(new_ecg_data)")

In [ ]:
# ============================================================
# CELL 25: Final summary printout
# ============================================================
# Print a clean final summary of everything computed in this notebook.

print('=' * 70)
print('FINAL RESULTS SUMMARY')
print('=' * 70)
print(f'Dataset   : PhysioNet 2017 AF Classification Challenge')
print(f'Recordings: {len(raw_signals):,} (after loading)')
print(f'Classes   : N (Normal), A (AF), O (Other), ~ (Noisy)')
print(f'Signal len: {TARGET_LEN} samples = 30 s at 300 Hz')
print(f'Train/Val/Test split: 70% / 15% / 15%')
print()
print(f'{'Model':<20} {'Test Accuracy':>14} {'Macro F1':>10} {'AF Recall':>10}')
print('-' * 60)
print(f'{'1D CNN':<20} {cnn_acc*100:>13.2f}% {cnn_macro_f1:>10.4f} {cnn_r[af_idx]:>10.4f}')
print(f'{'CNN-LSTM':<20} {lstm_acc*100:>13.2f}% {lstm_macro_f1:>10.4f} {lstm_r[af_idx]:>10.4f}')
print('=' * 70)
print()
print('Generated plots saved in current directory:')
for fname in ['class_distribution.png', 'sample_ecg_traces.png',
              'signal_length_distribution.png', 'cnn_training_curves.png',
              'lstm_training_curves.png', 'confusion_matrices.png',
              'f1_comparison.png']:
    exists = os.path.exists(fname)
    print(f'  [{"OK" if exists else "??" }] {fname}')